# 04 — Environmental Verification Experiment

**Question.** Can environmental claims be grounded in measurable Earth-observation
evidence rather than text alone — and when they cannot, does the system say so?

**What this notebook does NOT do.** It does not measure whether claims are *true*. There
is no adjudicated corpus of true/false flaring claims, so any accuracy number here would
rest on labels somebody invented. Instead it walks the full evidence chain for real
fields and records, at every step, what the observations could and could not support.

**Three things are kept strictly apart, and the notebook never blurs them:**

| | |
|---|---|
| **Observed physical evidence** | what the satellite series actually shows |
| **Reported claim** | what the text asserts |
| **Algorithmic consistency assessment** | whether the two are consistent, and how uncertain that assessment is |

**Probe claims.** Claims are constructed from real observed windows so the pipeline can
be exercised uniformly across all 12 fields. They are *probes* — nobody published them —
and they are labelled as such everywhere. Every observation is real.

| Notebook card | |
|---|---|
| **Type** | Analyse (no training) |
| **Purpose** | Walk the full evidence chain for every monitored field with probe claims built from real observed windows. Records what the evidence can support — not accuracy. |
| **Inputs** | The `greentruth` package and `data/real/` (from notebook 03). |
| **Outputs** | `evaluation/04_verification_per_field.csv`, `evaluation/04_verification_summary.json`. |
| **Where it runs** | Locally from a checkout (`jupyter` or `nbclient`, run from `notebooks/`), or Colab with the project folder available. |
| **Execution record** | Executed locally against the final code — record in `notebooks/executed/04_environmental_verification_experiment.ipynb`. |


## 0. Setup — repo and real data

In [ ]:
# ---------------------------------------------------------------- setup
# This notebook needs the GreenTruth package and the real data in data/real/.
# Nothing is uploaded by hand: if the repo is not present, the public flaring
# data is downloaded directly from the World Bank.
import os, sys, json, glob
from pathlib import Path

CANDIDATES = [".", "..", "/content", "/content/greentruth",
              "/content/drive/MyDrive/greentruth"]
REPO = None
for c in CANDIDATES:
    if os.path.isdir(os.path.join(c, "greentruth")):
        REPO = os.path.abspath(c)
        if REPO not in sys.path:
            sys.path.insert(0, REPO)
        break

if REPO is None:
    raise SystemExit(
        "GreenTruth package not found.\n"
        "Make it importable by ONE of:\n"
        "  * Colab Files panel -> upload the project folder\n"
        "  * from google.colab import drive; drive.mount('/content/drive')\n"
        "    with the project at /content/drive/MyDrive/greentruth\n"
        "  * run this notebook from inside a local checkout")

os.chdir(REPO)
print("repo:", REPO)

REAL = Path(REPO) / "data" / "real" / "flaring_by_field.csv"
if not REAL.exists():
    print("\ndata/real/flaring_by_field.csv missing -> run notebook 03 first.")
    print("It downloads the public World Bank release directly (no account).")
    raise SystemExit("real data not present")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from greentruth import GreenTruth

gt = GreenTruth(detector_preference="rule_based")
cov = gt.evidence.coverage()
print(f"\nreal evidence : {cov['n_observations']} annual observations, "
      f"{cov['n_fields']} fields, {cov['year_min']}-{cov['year_max']}")
print(f"methane       : {'loaded' if gt.methane.available else 'not loaded'}")
print(f"detector      : {gt.detector_info['detector']}"
      f"{' (fallback)' if gt.detector_info.get('fallback') else ''}")

In [ ]:
# ------------------------------------------------- data provenance banner
# Printed before any analysis so a reader can see exactly what is loaded,
# where it came from, and under what licence — without leaving the notebook.
reg_path = Path(REPO) / "data" / "dataset_registry.json"
if reg_path.exists():
    reg = json.loads(reg_path.read_text(encoding="utf-8"))
    print("=" * 78)
    print("DATA PROVENANCE")
    print("=" * 78)
    for d in reg["datasets"]:
        ff = d.get("file_facts") or {}
        if not ff.get("present") and not d.get("n_test"):
            continue
        if d.get("role_in_greentruth") == "NOT USED":
            continue
        print(f"\nDATA SOURCE   {d['name']}")
        print(f"PROVIDER      {d['provider']}")
        print(f"INSTRUMENT    {d['instrument']}")
        print(f"URL           {d['official_url']}")
        print(f"LICENSE       {d['licence']}")
        print(f"RESOLUTION    {d['spatial_resolution']} | {d['temporal_resolution']}")
        if ff.get("present"):
            print(f"DATA SHAPE    {ff['rows']:,} rows x {len(ff['columns'])} cols "
                  f"| {ff['entities']} entities | {ff['year_min']}-{ff['year_max']}")
        elif d.get("n_test"):
            print(f"DATA SHAPE    test split n = {d['n_test']}")
        print(f"DATE ACCESSED {d['date_accessed']}  ({d['date_accessed_note']})")
        print(f"ROLE          {d['role_in_greentruth']}")
    print("\n" + "=" * 78)
    print(reg["independence_rule"])
    print("=" * 78)
else:
    print("data/dataset_registry.json not found — run "
          "scripts/build_dataset_registry.py to generate it.")

## 1. The evidence chain, one claim at a time

For a single field we walk every stage and print what each one produced. This is the
same code path the application uses; nothing is reimplemented for the notebook.

In [ ]:
FIELD = "Niger Delta"
TEXT = "We reduced routine gas flaring by 40% by 2023 from 2012 levels."

res = gt.analyze(TEXT, FIELD)
claim = res["claims"][0]

print("=" * 76)
print("CLAIM ->", TEXT)
print("=" * 76)
cl = claim["claim"]
print(f"\n1. CLAIM STRUCTURE")
print(f"   type            {cl['claim_type_label']}")
print(f"   metric          {cl['metric']}  (observation channel: {cl['metric_supported']})")
print(f"   claimed change  {cl['claimed_change_percent']}%")
print(f"   window          {cl['baseline_year']} -> {cl['comparison_year']}")
print(f"   slot coverage   {cl['extract_confidence']} ({cl['extract_confidence_kind']})")

f = res["field"]
print(f"\n2. FACILITY")
print(f"   {f['name']} ({f['country']}) at {f['lat']}, {f['lon']}")
print(f"   match radius    {f['match_radius_km']} km")

ev = claim["evidence"]
print(f"\n3. EARTH OBSERVATION")
print(f"   {ev['dataset_name']}")
print(f"   {ev['measurement_type']}, {ev['temporal_resolution']}")
print(f"   {ev['years'][0]}-{ev['years'][-1]}, {len(ev['years'])} observations")

a = claim["analysis"]
print(f"\n4. TEMPORAL ANALYSIS")
print(f"   {a['baseline_value']} -> {a['comparison_value']} {ev['unit']}")
print(f"   observed        {a['observed_change']*100:+.1f}%")
print(f"   claimed         {a['claimed_change']*100:+.1f}%")

print(f"\n5. UNCERTAINTY")
print(f"   90% interval    {a['interval'][0]*100:+.0f}% to {a['interval'][1]*100:+.0f}%")
print(f"   decision regions spanned: {a['region_span']}")
ic = claim.get("interval_calibration") or {}
if ic:
    print(f"   MEASURED coverage at a {ic['year_gap']}-year gap: "
          f"{ic['measured_coverage_at_this_gap']} (notebook 05)")

sy = claim.get("synthesis") or {}
print(f"\n6. CROSS-SOURCE SYNTHESIS -> {sy.get('label')}")
for s in sy.get("sources", []):
    print(f"   {s['name']:18s} {s['signal']:>10s}  {s['role']}")

s = claim["sufficiency"]
print(f"\n7. EVIDENCE SUFFICIENCY -> {s['level']}")
for chk in s["checks"]:
    print(f"   {chk['symbol']} {chk['label']}")
print(f"   ceiling: {s['evidence_ceiling']}")

print(f"\n8. VERDICT -> {claim['verdict_label'].upper()}")
print("  ", claim["rationale"][:400])

## 2. Observed vs claim-implied trajectory

The **claim-implied trajectory** is a straight line from the baseline value to the value
the claim implies at the comparison year. It is arithmetic on the claim, **not** an
observation and **not** a causal counterfactual — it is what the field would look like if
the claim described it exactly.

In [ ]:
years = ev["years"]; values = ev["values"]
b, e = a["baseline_year"], a["comparison_year"]
bv = a["baseline_value"]
implied_end = bv * (1 + a["claimed_change"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(years, values, "o-", color="#15b879", lw=2.2, label="Observed (VIIRS)")
ax.plot([b, e], [bv, implied_end], "--", color="#f4b942", lw=2,
        label="Claim-implied trajectory (arithmetic, not an observation)")
ax.scatter([e], [implied_end], color="#f4b942", zorder=5)
ax.scatter([e], [a["comparison_value"]], color="#15b879", zorder=5)

lo, hi = a["interval"]
ax.fill_between([e - 0.18, e + 0.18], bv * (1 + lo), bv * (1 + hi),
                color="#1ea5a1", alpha=0.22,
                label=f"90% interval on the observed change")
for yr, lab in ((b, "baseline"), (e, "outcome")):
    ax.axvline(yr, color="#7c9aa1", ls=":", lw=1, alpha=.7)
    ax.text(yr, ax.get_ylim()[1] * 0.97, lab, ha="center", fontsize=9, color="#7c9aa1")

ax.set_xlabel("year"); ax.set_ylabel(ev["unit"])
ax.set_title(f"{f['name']}: observed vs claim-implied  |  verdict: {claim['verdict_label']}")
ax.legend(fontsize=9); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

print("Observed and claim-implied are drawn in different styles on purpose.")
print("The dashed line is what the CLAIM says; the solid line is what the SATELLITE saw.")

## 3. Every field, same protocol

The same probe construction as the application, across all 12 fields, over the longest
window each field supports. This is where the abstention behaviour becomes visible.

In [ ]:
rows = []
for fid, info in gt.evidence.fields.items():
    s = gt.evidence.series_for(fid)
    if not s:
        continue
    ys = sorted(s); b, e = ys[0], ys[-1]
    if not s[b]:
        continue
    obs = (s[e] - s[b]) / s[b]
    pct = max(5, int(round(abs(obs) * 100 / 5.0) * 5))
    direction = "reduced" if obs < 0 else "increased"
    text = f"We {direction} routine gas flaring by {pct}% by {e} from {b} levels."
    r = gt.analyze(text, info["name"])
    if not r["claims"]:
        continue
    cc = r["claims"][0]; aa = cc["analysis"]; ss = cc["sufficiency"]
    sy = cc.get("synthesis") or {}
    m = cc.get("methane") or {}
    rows.append(dict(
        field=info["name"], window=f"{b}-{e}",
        claimed=f"{-pct if obs < 0 else pct}%",
        observed=f"{aa['observed_change']*100:+.0f}%",
        interval=f"[{aa['interval'][0]*100:+.0f}%, {aa['interval'][1]*100:+.0f}%]",
        verdict=cc["verdict"], sufficiency=ss["level"],
        synthesis=sy.get("state"),
        methane=("yes" if m.get("available") else "no")))

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print("verdicts   :", df.verdict.value_counts().to_dict())
print("sufficiency:", df.sufficiency.value_counts().to_dict())
print("synthesis  :", df.synthesis.value_counts().to_dict())

## 4. Save results

In [ ]:
outdir = Path(REPO) / "evaluation"
outdir.mkdir(exist_ok=True)
df.to_csv(outdir / "04_verification_per_field.csv", index=False)

summary = dict(
    experiment="04_environmental_verification",
    n_fields=len(df),
    verdicts=df.verdict.value_counts().to_dict(),
    sufficiency_levels=df.sufficiency.value_counts().to_dict(),
    synthesis_states=df.synthesis.value_counts().to_dict(),
    methane_available=int((df.methane == "yes").sum()),
    integrity=dict(
        ground_truth_available=False,
        accuracy_reported=False,
        why=("No adjudicated true/false corpus exists for these claims. This "
             "experiment records what the evidence can support, not whether the "
             "claims are true."),
        probe_claims=("Claims are constructed from real observed windows to exercise "
                      "the pipeline. All observations are real.")))
(outdir / "04_verification_summary.json").write_text(json.dumps(summary, indent=2))
print("saved -> evaluation/04_verification_per_field.csv")
print("saved -> evaluation/04_verification_summary.json")
print()
print(json.dumps(summary["verdicts"], indent=2))